# 🩺 Prediksi Diabetes dengan Logistic Regression

Notebook ini mendemonstrasikan penggunaan **Logistic Regression** untuk memprediksi kemungkinan seseorang menderita diabetes berdasarkan beberapa fitur kesehatan.

**Peningkatan pada versi ini:**
- 💾 Dataset yang dibuat disimpan ke folder `data/` (CSV)
- 🤖 Model & scaler disimpan ke folder `models/` (`.pkl`)
- 📊 Setiap analisis dilengkapi visualisasi (Matplotlib & Seaborn) yang disimpan sebagai gambar PNG ke folder `gambar/`

## 📋 Daftar Isi
1. Install & Import Library
2. Penyiapan Dataset (+ simpan dataset)
3. Eksplorasi Data (+ visualisasi)
4. Pemisahan Data Train & Test (+ simpan)
5. Standarisasi Fitur
6. Pelatihan Model (+ learning curve)
7. Evaluasi Model (+ visualisasi)
8. Cross-Validation (+ visualisasi)
9. Pengujian Tambahan & Matriks Evaluasi Lanjutan (+ visualisasi)
10. Menyimpan Model (.pkl) & Hasil
11. Memuat Kembali Model & Prediksi

In [ ]:
!pip install -r requirements.txt

## 1️⃣ Install & Import Library

Pada langkah ini kita mengimpor semua library yang dibutuhkan:
- **NumPy**: komputasi numerik
- **Pandas**: manipulasi data
- **Matplotlib & Seaborn**: visualisasi data
- **Scikit-learn**: model machine learning dan metrik evaluasi
- **Joblib**: menyimpan dan memuat model (`.pkl`)

Kita juga membuat tiga folder output: `data/`, `models/`, dan `gambar/`.

In [ ]:
import os
import json
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

# ---------- Folder output ----------
DATA_DIR   = 'data'      # dataset & hasil prediksi (CSV)
MODEL_DIR  = 'models'    # model & scaler (.pkl)
GAMBAR_DIR = 'gambar'    # semua visualisasi (PNG)
for folder in (DATA_DIR, MODEL_DIR, GAMBAR_DIR):
    os.makedirs(folder, exist_ok=True)

# ---------- Gaya visualisasi ----------
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

LABEL = {0: 'Tidak Diabetes', 1: 'Diabetes'}
PALET = {'Tidak Diabetes': '#4C9BE8', 'Diabetes': '#E8574C'}

def simpan_gambar(nama_file):
    """Simpan figure aktif ke folder gambar/ lalu tampilkan."""
    path = os.path.join(GAMBAR_DIR, nama_file)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print(f"✅ Gambar disimpan: {path}")
    plt.show()

print("✅ Semua library berhasil diimpor!")
print(f"📁 Folder output: {DATA_DIR}/, {MODEL_DIR}/, {GAMBAR_DIR}/")

## 2️⃣ Penyiapan Dataset

Karena dataset asli tidak disediakan, kita akan **membuat dataset sintetis** dengan `numpy.random` yang menyerupai dataset *Pima Indians Diabetes*.

### Fitur yang digunakan:
| Fitur | Deskripsi |
|-------|-----------|
| `pregnancies` | Jumlah kehamilan |
| `glucose` | Kadar glukosa plasma |
| `blood_pressure` | Tekanan darah diastolik |
| `bmi` | Body Mass Index |
| `age` | Usia |
| `outcome` | Target (1 = diabetes, 0 = tidak) |

Dataset yang dibuat langsung **disimpan ke `data/diabetes_dataset.csv`**.

In [ ]:
# Set seed agar hasil reproducible
np.random.seed(42)
n_samples = 600

# Membuat fitur-fitur sintetis
pregnancies    = np.random.randint(0, 10, n_samples)
glucose        = np.random.normal(120, 35, n_samples).clip(70, 200)
blood_pressure = np.random.normal(70, 15, n_samples).clip(40, 110)
bmi            = np.random.normal(28, 7, n_samples).clip(15, 60)
age            = np.random.randint(21, 70, n_samples)

# Membuat target (outcome) berdasarkan kombinasi fitur + noise
logit = (-8 + 0.04 * glucose + 0.05 * bmi + 0.02 * blood_pressure 
         + 0.01 * age + 0.05 * pregnancies)
prob  = 1 / (1 + np.exp(-logit))
outcome = (np.random.rand(n_samples) < prob).astype(int)

# Membentuk DataFrame
df = pd.DataFrame({
    'pregnancies': pregnancies,
    'glucose': glucose,
    'blood_pressure': blood_pressure,
    'bmi': bmi,
    'age': age,
    'outcome': outcome
})

print(f"✅ Dataset berhasil dibuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom")

# ---------- Simpan dataset ke CSV ----------
path_dataset = os.path.join(DATA_DIR, 'diabetes_dataset.csv')
df.to_csv(path_dataset, index=False)
print(f"✅ Dataset berhasil disimpan: {path_dataset} ({os.path.getsize(path_dataset) / 1024:.1f} KB)")

# Verifikasi: baca kembali file yang baru disimpan
df_cek = pd.read_csv(path_dataset)
print(f"🔎 Verifikasi baca ulang: {df_cek.shape[0]} baris, {df_cek.shape[1]} kolom")
df.head()

## 3️⃣ Eksplorasi Data (EDA)

Sebelum melatih model, kita periksa:
- Distribusi kelas target
- Statistik deskriptif
- Nilai yang hilang (missing values)

In [ ]:
# Distribusi kelas target
print("Distribusi kelas 'outcome':")
print(df['outcome'].value_counts(normalize=True).round(3))
print()

# Statistik deskriptif
print("Statistik deskriptif:")
print(df.describe().round(2))
print()

# Cek missing values
print(f"Total missing values: {df.isnull().sum().sum()}")

## 3.1 Visualisasi Eksplorasi Data

Empat visualisasi untuk memahami data:
1. **Distribusi kelas** → apakah data seimbang?
2. **Histogram + KDE** per fitur → bagaimana sebaran tiap fitur pada masing-masing kelas?
3. **Boxplot** per fitur → perbandingan median & outlier antar kelas
4. **Heatmap korelasi** → hubungan antar fitur dan terhadap target

In [ ]:
# Data khusus untuk plotting (label kelas dalam teks)
df_plot = df.assign(status=df['outcome'].map(LABEL))
fitur = [c for c in df.columns if c != 'outcome']

# --- 1) Distribusi kelas ---
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.countplot(data=df_plot, x='status', hue='status', order=list(PALET),
              palette=PALET, legend=False, ax=ax)
total = len(df_plot)
for bar in ax.patches:
    h = bar.get_height()
    ax.annotate(f"{int(h)}\n({h / total:.1%})", (bar.get_x() + bar.get_width() / 2, h),
                ha='center', va='bottom', fontsize=10)
ax.set_title('Distribusi Kelas Target (Outcome)')
ax.set_xlabel('')
ax.set_ylabel('Jumlah Sampel')
ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
fig.tight_layout()
simpan_gambar('01_distribusi_kelas.png')

In [ ]:
# --- 2) Histogram + KDE tiap fitur per kelas ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for i, col in enumerate(fitur):
    sns.histplot(data=df_plot, x=col, hue='status', palette=PALET, kde=True,
                 stat='density', common_norm=False, element='step',
                 hue_order=list(PALET), legend=(i == 0), ax=axes[i])
    axes[i].set_title(f'Distribusi {col}')
axes[-1].axis('off')   # subplot ke-6 tidak dipakai
fig.suptitle('Distribusi Fitur Berdasarkan Kelas', fontsize=14, y=1.01)
fig.tight_layout()
simpan_gambar('02_histogram_fitur.png')

In [ ]:
# --- 3) Boxplot tiap fitur per kelas ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for i, col in enumerate(fitur):
    sns.boxplot(data=df_plot, x='status', y=col, hue='status', order=list(PALET),
                palette=PALET, legend=False, ax=axes[i])
    axes[i].set_title(f'{col} vs Outcome')
    axes[i].set_xlabel('')
axes[-1].axis('off')
fig.suptitle('Boxplot Fitur Berdasarkan Kelas', fontsize=14, y=1.01)
fig.tight_layout()
simpan_gambar('03_boxplot_fitur.png')

In [ ]:
# --- 4) Heatmap korelasi ---
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Heatmap Korelasi Antar Variabel')
fig.tight_layout()
simpan_gambar('04_heatmap_korelasi.png')

## 4️⃣ Pemisahan Data Train & Test

Kita membagi data menjadi:
- **80% Training set** → untuk melatih model
- **20% Testing set** → untuk menguji performa model

Parameter `stratify=y` memastikan proporsi kelas tetap seimbang di kedua set.

Data train dan test juga **disimpan ke `data/train.csv` dan `data/test.csv`**.

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df.drop('outcome', axis=1)
y = df['outcome']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Ukuran data training : {X_train.shape}")
print(f"Ukuran data testing  : {X_test.shape}")

# Simpan hasil pemisahan data
train_path = os.path.join(DATA_DIR, 'train.csv')
test_path  = os.path.join(DATA_DIR, 'test.csv')
X_train.assign(outcome=y_train).to_csv(train_path, index=False)
X_test.assign(outcome=y_test).to_csv(test_path, index=False)
print(f"✅ Data training disimpan: {train_path}")
print(f"✅ Data testing disimpan : {test_path}")

In [ ]:
# Visualisasi: proporsi kelas pada train vs test
prop = pd.DataFrame({
    'Training': y_train.value_counts(normalize=True).sort_index(),
    'Testing' : y_test.value_counts(normalize=True).sort_index()
}).rename(index=LABEL)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Ukuran dataset
axes[0].bar(['Training', 'Testing'], [len(X_train), len(X_test)], color=['#6C8EBF', '#F0A04B'])
for i, v in enumerate([len(X_train), len(X_test)]):
    axes[0].text(i, v, str(v), ha='center', va='bottom')
axes[0].set_title('Jumlah Sampel (80% : 20%)')
axes[0].set_ylabel('Jumlah Sampel')

# Proporsi kelas (membuktikan stratify bekerja)
prop.T.plot(kind='bar', ax=axes[1], color=[PALET[c] for c in prop.index], rot=0)
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt='%.3f', padding=2)
axes[1].set_title('Proporsi Kelas per Set (stratify=y)')
axes[1].set_ylabel('Proporsi')
axes[1].set_ylim(0, 1)
axes[1].legend(title='')

fig.tight_layout()
simpan_gambar('05_split_train_test.png')

## 5️⃣ Standarisasi Fitur

Logistic Regression sensitif terhadap skala fitur. Kita gunakan `StandardScaler` untuk menormalkan data (mean=0, std=1).

> ⚠️ **Penting:** Scaler hanya di-*fit* pada training set, lalu di-*transform* ke test set untuk menghindari **data leakage**.

In [ ]:
scaler = StandardScaler()

# Fit pada training, transform ke training & test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Standarisasi selesai!")
print(f"Mean data training setelah scaling : {X_train_scaled.mean(axis=0).round(3)}")
print(f"Std  data training setelah scaling : {X_train_scaled.std(axis=0).round(3)}")

In [ ]:
# Visualisasi: sebelum vs sesudah standarisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=X_train, orient='h', color='#9DB8D9', ax=axes[0])
axes[0].set_title('Sebelum Standarisasi (skala berbeda-beda)')
axes[0].set_xlabel('Nilai asli')

sns.boxplot(data=pd.DataFrame(X_train_scaled, columns=X.columns), orient='h',
            color='#A8D5A2', ax=axes[1])
axes[1].axvline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Sesudah Standarisasi (mean=0, std=1)')
axes[1].set_xlabel('Nilai terstandarisasi (z-score)')

fig.tight_layout()
simpan_gambar('06_standarisasi_fitur.png')

## 6️⃣ Pelatihan Model Logistic Regression

Kita melatih model Logistic Regression dengan parameter:
- `max_iter=1000` → iterasi maksimum agar konvergen
- `random_state=42` → hasil reproducible

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Prediksi
y_pred      = model.predict(X_test_scaled)
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

print("✅ Model berhasil dilatih dan prediksi dibuat!")
print(f"Akurasi training : {model.score(X_train_scaled, y_train):.4f}")
print(f"Akurasi testing  : {model.score(X_test_scaled, y_test):.4f}")

### 6.1 Learning Curve

Learning curve menunjukkan apakah model mengalami **overfitting** (skor training jauh lebih tinggi dari validasi) atau **underfitting** (kedua skor rendah) seiring bertambahnya jumlah data latih.

In [ ]:
train_sizes, train_sc, val_sc = learning_curve(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_scaled, y_train,
    cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8), random_state=42, shuffle=True
)

fig, ax = plt.subplots(figsize=(8, 5))
for sc, warna, nama in [(train_sc, '#4C9BE8', 'Skor Training'), (val_sc, '#E8574C', 'Skor Validasi (CV)')]:
    mean, std = sc.mean(axis=1), sc.std(axis=1)
    ax.plot(train_sizes, mean, 'o-', color=warna, label=nama)
    ax.fill_between(train_sizes, mean - std, mean + std, color=warna, alpha=0.15)
ax.set_title('Learning Curve - Logistic Regression')
ax.set_xlabel('Jumlah Data Latih')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
fig.tight_layout()
simpan_gambar('07_learning_curve.png')

## 7️⃣ Evaluasi Model

Kita mengevaluasi performa model menggunakan beberapa metrik:

| Metrik | Deskripsi |
|--------|-----------|
| **Accuracy** | Persentase prediksi benar |
| **Precision** | Dari yang diprediksi positif, berapa yang benar |
| **Recall** | Dari yang sebenarnya positif, berapa yang tertangkap |
| **F1-Score** | Harmonic mean dari Precision & Recall |
| **ROC-AUC** | Kemampuan model membedakan antar kelas |
| **Confusion Matrix** | Tabel TP, TN, FP, FN |

In [ ]:
# Hitung semua metrik
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred)

# Tampilkan hasil
print("=" * 45)
print("       HASIL EVALUASI MODEL")
print("=" * 45)
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")
print("=" * 45)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, 
    index=['Aktual: 0', 'Aktual: 1'],
    columns=['Prediksi: 0', 'Prediksi: 1']))

## 7.1 Visualisasi Evaluasi Model

Visualisasi evaluasi (semuanya disimpan sebagai PNG):
1. **Confusion matrix** (jumlah & persentase per baris)
2. **Kurva ROC** dan **Precision-Recall**
3. **Distribusi probabilitas prediksi** per kelas aktual dan **ringkasan metrik**

In [ ]:
# --- 1) Confusion Matrix: jumlah & dinormalisasi per baris ---
nama_kelas = [LABEL[0], LABEL[1]]
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=nama_kelas, yticklabels=nama_kelas, ax=axes[0])
axes[0].set_title('Confusion Matrix (Jumlah)')
axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', cbar=False,
            xticklabels=nama_kelas, yticklabels=nama_kelas, ax=axes[1])
axes[1].set_title('Confusion Matrix (Persentase per Baris Aktual)')
axes[1].set_xlabel('Prediksi'); axes[1].set_ylabel('Aktual')

fig.tight_layout()
simpan_gambar('08_confusion_matrix.png')

In [ ]:
# --- 2) Kurva ROC dan Precision-Recall ---
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_pred_prob)
ap = average_precision_score(y_test, y_pred_prob)
baseline = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='#E8574C', linewidth=2, label=f'Logistic Regression (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Tebakan acak (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#E8574C')
axes[0].set_title('Kurva ROC')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right')

axes[1].plot(rec_curve, prec_curve, color='#4C9BE8', linewidth=2, label=f'Logistic Regression (AP = {ap:.3f})')
axes[1].axhline(baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.2f})')
axes[1].set_title('Kurva Precision-Recall')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='upper right')

fig.tight_layout()
simpan_gambar('09_roc_dan_precision_recall.png')

In [ ]:
# --- 3) Distribusi probabilitas prediksi & ringkasan metrik ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_prob = pd.DataFrame({'prob': y_pred_prob, 'aktual': y_test.map(LABEL).values})
sns.histplot(data=df_prob, x='prob', hue='aktual', palette=PALET, bins=20,
             element='step', hue_order=list(PALET), ax=axes[0])
axes[0].axvline(0.5, color='k', linestyle='--', label='Threshold 0.5')
axes[0].set_title('Distribusi Probabilitas Prediksi per Kelas Aktual')
axes[0].set_xlabel('Probabilitas diprediksi diabetes')
axes[0].set_ylabel('Jumlah Sampel')

nama_metrik = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
nilai_metrik = [acc, prec, rec, f1, auc]
bars = axes[1].bar(nama_metrik, nilai_metrik, color=sns.color_palette('viridis', len(nama_metrik)))
axes[1].bar_label(bars, fmt='%.3f', padding=3)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Ringkasan Metrik Evaluasi (Data Test)')
axes[1].set_ylabel('Skor')

fig.tight_layout()
simpan_gambar('10_probabilitas_dan_metrik.png')

## 8️⃣ Cross-Validation

Untuk memastikan performa model **stabil** dan tidak bergantung pada satu split tertentu, kita lakukan **5-Fold Cross-Validation** menggunakan seluruh data training.

In [ ]:
cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_scaled, y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross-Validation Accuracy (5-Fold):")
print(f"  Skor tiap fold : {cv_scores.round(4)}")
print(f"  Rata-rata      : {cv_scores.mean():.4f}")
print(f"  Standar deviasi: {cv_scores.std():.4f}")

In [ ]:
# Visualisasi skor cross-validation
fig, ax = plt.subplots(figsize=(8, 5))
folds = [f'Fold {i}' for i in range(1, len(cv_scores) + 1)]
bars = ax.bar(folds, cv_scores, color='#6C8EBF')
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.axhline(cv_scores.mean(), color='#E8574C', linestyle='--', linewidth=2,
           label=f'Rata-rata = {cv_scores.mean():.4f}')
ax.axhspan(cv_scores.mean() - cv_scores.std(), cv_scores.mean() + cv_scores.std(),
           color='#E8574C', alpha=0.12, label=f'±1 Std = {cv_scores.std():.4f}')
ax.set_ylim(0, 1.1)
ax.set_title('Skor Accuracy 5-Fold Cross-Validation')
ax.set_ylabel('Accuracy')
ax.legend(loc='lower right')
fig.tight_layout()
simpan_gambar('11_cross_validation.png')

## 9️⃣ Pengujian Tambahan & Matriks Evaluasi Lanjutan

Bagian ini melengkapi evaluasi dengan:
- **Classification report** per kelas
- **Specificity, Balanced Accuracy, dan MCC** (berguna jika kelas tidak seimbang)
- **Analisis threshold** untuk melihat trade-off Precision vs Recall
- **Koefisien model** untuk melihat fitur yang paling berpengaruh
- **Prediksi pasien baru** sebagai contoh penggunaan model

Setiap analisis disertai visualisasinya.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, matthews_corrcoef

# Classification report per kelas
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Diabetes', 'Diabetes'], zero_division=0))

# Metrik lanjutan dari confusion matrix
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
bal_acc     = balanced_accuracy_score(y_test, y_pred)
mcc         = matthews_corrcoef(y_test, y_pred)

print("=" * 45)
print("       METRIK LANJUTAN")
print("=" * 45)
print(f"True Positive (TP)  : {tp}")
print(f"True Negative (TN)  : {tn}")
print(f"False Positive (FP) : {fp}")
print(f"False Negative (FN) : {fn}")
print("-" * 45)
print(f"Specificity         : {specificity:.4f}")
print(f"Balanced Accuracy   : {bal_acc:.4f}")
print(f"Matthews Corr. Coef : {mcc:.4f}")

In [ ]:
# Visualisasi: metrik lanjutan & komponen confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nama = ['Precision', 'Recall (Sensitivity)', 'Specificity', 'F1-Score', 'Balanced Acc.', 'MCC']
nilai = [prec, rec, specificity, f1, bal_acc, mcc]
bars = axes[0].barh(nama, nilai, color=sns.color_palette('mako_r', len(nama)))
axes[0].bar_label(bars, fmt='%.3f', padding=3)
axes[0].set_xlim(min(0, min(nilai) - 0.05), 1.1)
axes[0].invert_yaxis()
axes[0].set_title('Metrik Evaluasi Lanjutan')
axes[0].set_xlabel('Skor')

komponen = ['TP', 'TN', 'FP', 'FN']
jumlah = [tp, tn, fp, fn]
warna = ['#4CAF50', '#4CAF50', '#E8574C', '#E8574C']
bars = axes[1].bar(komponen, jumlah, color=warna)
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Komponen Confusion Matrix (hijau = benar, merah = salah)')
axes[1].set_ylabel('Jumlah Sampel')

fig.tight_layout()
simpan_gambar('12_metrik_lanjutan.png')

In [ ]:
# Analisis threshold: bagaimana metrik berubah jika ambang batas diubah?
rows = []
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_thr = (y_pred_prob >= thr).astype(int)
    rows.append({
        'threshold': thr,
        'accuracy' : accuracy_score(y_test, y_thr),
        'precision': precision_score(y_test, y_thr, zero_division=0),
        'recall'   : recall_score(y_test, y_thr, zero_division=0),
        'f1'       : f1_score(y_test, y_thr, zero_division=0)
    })

print("Analisis Threshold:")
pd.DataFrame(rows).round(4)

In [ ]:
# Visualisasi analisis threshold (grid lebih halus)
thr_grid = np.arange(0.05, 0.96, 0.025)
hasil_thr = pd.DataFrame({
    'threshold': thr_grid,
    'Accuracy' : [accuracy_score(y_test, (y_pred_prob >= t).astype(int)) for t in thr_grid],
    'Precision': [precision_score(y_test, (y_pred_prob >= t).astype(int), zero_division=0) for t in thr_grid],
    'Recall'   : [recall_score(y_test, (y_pred_prob >= t).astype(int), zero_division=0) for t in thr_grid],
    'F1-Score' : [f1_score(y_test, (y_pred_prob >= t).astype(int), zero_division=0) for t in thr_grid],
})
thr_terbaik = hasil_thr.loc[hasil_thr['F1-Score'].idxmax(), 'threshold']

fig, ax = plt.subplots(figsize=(9, 5.5))
for kolom, warna in zip(['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                        ['#6C8EBF', '#F0A04B', '#4CAF50', '#E8574C']):
    ax.plot(hasil_thr['threshold'], hasil_thr[kolom], label=kolom, color=warna, linewidth=2)
ax.axvline(0.5, color='gray', linestyle=':', label='Threshold default (0.5)')
ax.axvline(thr_terbaik, color='k', linestyle='--', label=f'F1 terbaik (threshold = {thr_terbaik:.3f})')
ax.set_title('Pengaruh Threshold terhadap Metrik')
ax.set_xlabel('Threshold probabilitas')
ax.set_ylabel('Skor')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left')
fig.tight_layout()
simpan_gambar('13_analisis_threshold.png')

In [ ]:
# Koefisien model (fitur sudah distandarisasi, jadi bisa dibandingkan langsung)
coef_df = pd.DataFrame({
    'fitur'      : X.columns,
    'koefisien'  : model.coef_[0],
    'odds_ratio' : np.exp(model.coef_[0])
}).sort_values('koefisien', ascending=False)

print(f"Intercept: {model.intercept_[0]:.4f}\n")
coef_df.round(4)

In [ ]:
# Visualisasi koefisien & odds ratio
urut = coef_df.sort_values('koefisien')
warna = ['#4C9BE8' if k < 0 else '#E8574C' for k in urut['koefisien']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].barh(urut['fitur'], urut['koefisien'], color=warna)
axes[0].bar_label(bars, fmt='%.3f', padding=3)
axes[0].axvline(0, color='k', linewidth=1)
axes[0].set_title('Koefisien Logistic Regression\n(merah = menaikkan risiko, biru = menurunkan)')
axes[0].set_xlabel('Koefisien (fitur terstandarisasi)')

bars = axes[1].barh(urut['fitur'], urut['odds_ratio'], color=warna)
axes[1].bar_label(bars, fmt='%.2f', padding=3)
axes[1].axvline(1, color='k', linestyle='--', linewidth=1)
axes[1].set_title('Odds Ratio per +1 Std Fitur\n(garis putus-putus = tidak berpengaruh)')
axes[1].set_xlabel('Odds Ratio')

fig.tight_layout()
simpan_gambar('14_koefisien_model.png')

In [ ]:
# Contoh prediksi untuk pasien baru
pasien_baru = pd.DataFrame({
    'pregnancies'   : [2, 6],
    'glucose'       : [95, 165],
    'blood_pressure': [68, 85],
    'bmi'           : [22.5, 34.0],
    'age'           : [28, 55]
})

pasien_scaled = scaler.transform(pasien_baru)
prob_baru = model.predict_proba(pasien_scaled)[:, 1]
pred_baru = model.predict(pasien_scaled)

hasil = pasien_baru.copy()
hasil['prob_diabetes'] = prob_baru.round(4)
hasil['prediksi'] = np.where(pred_baru == 1, 'Diabetes', 'Tidak Diabetes')
hasil

In [ ]:
# Visualisasi prediksi pasien baru
fig, ax = plt.subplots(figsize=(7, 5))
nama_pasien = [f'Pasien {i + 1}' for i in range(len(prob_baru))]
warna = [PALET['Diabetes'] if p >= 0.5 else PALET['Tidak Diabetes'] for p in prob_baru]
bars = ax.bar(nama_pasien, prob_baru, color=warna)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.axhline(0.5, color='k', linestyle='--', label='Threshold 0.5')
ax.set_ylim(0, 1.1)
ax.set_title('Probabilitas Diabetes untuk Pasien Baru')
ax.set_ylabel('Probabilitas')
ax.legend()
fig.tight_layout()
simpan_gambar('15_prediksi_pasien_baru.png')

## 🔟 Menyimpan Model (.pkl) & Hasil

Menyimpan model dan scaler dalam format **`.pkl`** agar dapat digunakan di masa depan tanpa perlu melatih ulang. Selain itu kita simpan:
- `models/info_model.json` → daftar fitur (urutan kolom), metrik, dan versi library
- `data/hasil_prediksi_test.csv` → prediksi model pada data test
- `data/metrik_evaluasi.csv` → ringkasan seluruh metrik

In [ ]:
# ---------- Simpan model & scaler (.pkl) ----------
path_model  = os.path.join(MODEL_DIR, 'model_diabetes.pkl')
path_scaler = os.path.join(MODEL_DIR, 'scaler_diabetes.pkl')
joblib.dump(model, path_model)
joblib.dump(scaler, path_scaler)
print(f"✅ Model berhasil disimpan  : {path_model} ({os.path.getsize(path_model) / 1024:.1f} KB)")
print(f"✅ Scaler berhasil disimpan : {path_scaler} ({os.path.getsize(path_scaler) / 1024:.1f} KB)")

# ---------- Simpan info model (fitur, metrik, versi) ----------
metrik = {
    'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1,
    'roc_auc': auc, 'specificity': specificity, 'balanced_accuracy': bal_acc, 'mcc': mcc,
    'cv_accuracy_mean': float(cv_scores.mean()), 'cv_accuracy_std': float(cv_scores.std())
}
info_model = {
    'fitur': list(X.columns),
    'threshold_default': 0.5,
    'metrik': {k: round(float(v), 4) for k, v in metrik.items()},
    'versi_scikit_learn': sklearn.__version__
}
path_info = os.path.join(MODEL_DIR, 'info_model.json')
with open(path_info, 'w', encoding='utf-8') as f:
    json.dump(info_model, f, indent=2, ensure_ascii=False)
print(f"✅ Info model disimpan      : {path_info}")

# ---------- Simpan hasil prediksi data test & metrik ----------
hasil_test = X_test.copy()
hasil_test['aktual']        = y_test.values
hasil_test['prob_diabetes'] = y_pred_prob.round(4)
hasil_test['prediksi']      = y_pred
path_hasil = os.path.join(DATA_DIR, 'hasil_prediksi_test.csv')
hasil_test.to_csv(path_hasil, index=False)
print(f"✅ Hasil prediksi disimpan  : {path_hasil}")

path_metrik = os.path.join(DATA_DIR, 'metrik_evaluasi.csv')
pd.DataFrame({'metrik': list(info_model['metrik']), 'nilai': list(info_model['metrik'].values())}).to_csv(path_metrik, index=False)
print(f"✅ Metrik evaluasi disimpan : {path_metrik}")

## 1️⃣1️⃣ Memuat Kembali Model & Prediksi

Contoh cara memakai file `.pkl` yang sudah disimpan (misalnya di notebook atau script lain). Kita juga memverifikasi bahwa model yang dimuat memberi hasil **identik** dengan model di memori.

In [ ]:
# Muat model, scaler, dan info
model_muat  = joblib.load(os.path.join(MODEL_DIR, 'model_diabetes.pkl'))
scaler_muat = joblib.load(os.path.join(MODEL_DIR, 'scaler_diabetes.pkl'))
with open(os.path.join(MODEL_DIR, 'info_model.json'), encoding='utf-8') as f:
    info = json.load(f)

# Muat dataset test dari CSV
data_uji = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
X_uji = data_uji[info['fitur']]          # urutan kolom sesuai saat pelatihan

# Verifikasi: hasil model yang dimuat == model di memori
prob_muat = model_muat.predict_proba(scaler_muat.transform(X_uji))[:, 1]
print("Prediksi identik dengan model di memori:", np.allclose(prob_muat, y_pred_prob))

# Contoh prediksi satu pasien baru
pasien = pd.DataFrame({
    'pregnancies': [3], 'glucose': [140], 'blood_pressure': [75], 'bmi': [30.0], 'age': [45]
})[info['fitur']]
p = model_muat.predict_proba(scaler_muat.transform(pasien))[0, 1]
print(f"\nProbabilitas diabetes pasien contoh: {p:.4f}")
print(f"Prediksi: {'Diabetes' if p >= info['threshold_default'] else 'Tidak Diabetes'}")

In [ ]:
# Ringkasan semua file yang dihasilkan notebook ini
print("📦 FILE YANG DIHASILKAN")
print("=" * 55)
for folder in (DATA_DIR, MODEL_DIR, GAMBAR_DIR):
    print(f"\n📁 {folder}/")
    for nama in sorted(os.listdir(folder)):
        ukuran = os.path.getsize(os.path.join(folder, nama)) / 1024
        print(f"   - {nama:<38} {ukuran:8.1f} KB")

## ✅ Kesimpulan

- Model Logistic Regression berhasil dilatih pada dataset sintetis dan dievaluasi dengan berbagai metrik.
- Cross-validation menunjukkan seberapa stabil performa model pada berbagai subset data.
- Analisis threshold membantu memilih ambang batas sesuai kebutuhan (misalnya, memprioritaskan **recall** pada skrining medis agar lebih sedikit kasus terlewat).
- Koefisien model menunjukkan fitur yang paling berpengaruh terhadap prediksi.
- Dataset, data train/test, model, scaler, hasil prediksi, dan seluruh visualisasi tersimpan rapi di folder `data/`, `models/`, dan `gambar/`.

> ⚠️ **Catatan:** Dataset ini bersifat sintetis untuk tujuan pembelajaran dan **tidak boleh digunakan untuk diagnosis medis nyata**.